# 带时间窗与固定休息的容量约束车辆路径问题

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-time-windows-and-regular-breaks](https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-regular-breaks)


## 问题描述

**在带时间窗与固定休息的容量约束车辆路径问题**中,一组具有相同容量的配送车辆必须为客户提供服务。客户具有已知的营业时间以及对单一商品的需求。车辆从一个共同的配送中心出发并返回,且必须为驾驶员安排固定休息。目标是最小化总延误、所用车辆数量以及总行驶距离。

	

### 学习要点

- 添加 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 以建模每辆卡车的客户序列
- 添加 [integer decision variables](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#int) 以建模两次休息之间的时间间隔
- 使用 [recursive lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 定义数组,以计算客户的访问时间与驾驶员的休息开始时间
- 添加[多目标](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html),并将延误建模为[软约束](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)


## 数据

我们提供的带时间窗与固定休息的车辆路径问题算例来自 [Solomon 算例](http://web.cba.neu.edu/~msolomon/problems.htm)。数据文件的格式如下:

- 第一行给出算例的名称
- 第五行包含车辆数量及其公共容量
- 从第 10 行起,每个客户(从配送中心开始):

- 客户的索引
- x 坐标
- y 坐标
- 需求
- 最早到达时间
- 最晚到达时间
- 服务时间


## 建模方法

带时间窗与固定休息的容量约束车辆路径问题的 Hexaly 模型在 [CVRPTW 模型](https://www.hexaly.com/templates/vehicle-routing-problem-with-time-windows-cvrptw)的基础上扩展得到。关于该问题的路径与时间窗部分,我们请读者参阅该模型。

为了对此建模,我们引入了整型决策变量,表示每辆卡车相邻休息之间的时间间隔。通过以休息频率作为这些决策的上界,我们确保休息在整个规划时段内均匀分布。实际的休息时间则通过对这些间隔进行累积求和得到。

将休息纳入路径时间安排遵循以下原则:无论休息发生在行驶段、等待期还是服务期间,其固定时长都会被加到当前时间,从而使路径上的所有后续事件相应延后。

最后,目标与 CVRPTW 相同:我们按字典序依次最小化总延误、所用车辆数量以及总行驶距离。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


# Breaks parameters
# A break of 15 minutes every 4 hours
BREAKFREQUENCY = 60*4 # In minutes
BREAKDURATION = 15   # In minutes

def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def main(instance_file, str_time_limit, output_file):
    #
    # Read instance data
    #
    nb_customers, nb_trucks, truck_capacity, dist_matrix_data, dist_depot_data, \
        demands_data, service_time_data, earliest_start_data, latest_end_data, \
        max_horizon = read_input_cvrptwrb(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:

        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Sequence of customers visited by each truck
        customers_sequences = [model.list(nb_customers) for k in range(nb_trucks)]

        # All customers must be visited by exactly one truck
        model.constraint(model.partition(customers_sequences))

        # Create Hexaly arrays to be able to access them with an "at" operator
        demands = model.array(demands_data)
        earliest = model.array(earliest_start_data)
        latest = model.array(latest_end_data)
        service_time = model.array(service_time_data)
        dist_matrix = model.array(dist_matrix_data)
        dist_depot = model.array(dist_depot_data)

        dist_routes = [None] * nb_trucks
        end_time = [None] * nb_trucks
        home_lateness = [None] * nb_trucks
        lateness = [None] * nb_trucks
        trucks_used = [None] * nb_trucks

        # Number of breaks
        nb_breaks = int(math.ceil(max_horizon / BREAKFREQUENCY) + 1)

        # Time between the end of one break and the start of the next
        breaks_gaps = [[model.int(1, BREAKFREQUENCY) for _ in range(nb_breaks)] for _ in range(nb_trucks)]

        # Starting time of each break
        breaks_start_times = [None] * nb_trucks
        for k in range(0, nb_trucks):
            breaks_start_times_truck = [None] * nb_breaks
            for b in range(0, nb_breaks):
                breaks_start_times_truck[b] = model.sum(breaks_gaps[k][breakIdx] for breakIdx in range(b+1))
                + BREAKDURATION * b
                breaks_start_times[k] = breaks_start_times_truck

        for k in range(nb_trucks):
            sequence = customers_sequences[k]
            c = model.count(sequence)

            # A truck is used if it visits at least one customer
            trucks_used[k] = model.gt(c,0)

            # The quantity needed in each route must not exceed the truck capacity
            demand_lambda = model.lambda_function(lambda j: demands[j])
            route_quantity = model.sum(sequence, demand_lambda)
            model.constraint(route_quantity <= truck_capacity)

            # Distance traveled by each truck
            dist_lambda = model.lambda_function(
                lambda i: model.at(dist_matrix, sequence[i - 1], sequence[i]))
            dist_routes[k] = model.sum(model.range(1, c), dist_lambda) \
                + model.iif(c > 0, dist_depot[sequence[0]] + dist_depot[sequence[c - 1]], 0)
                       # Breaks must cover the entire horizon
            model.constraint(model.geq(breaks_start_times[k][nb_breaks-1], max_horizon + 1))

            # End of each visit
            end_time_lambda = model.lambda_function(
                lambda i, prev:
                    waiting_and_service_end(k, sequence[i],
                                        travel_end(k, i, prev, customers_sequences,model, dist_depot, dist_matrix, nb_breaks, breaks_start_times),
                                        model, earliest, service_time, nb_breaks, breaks_start_times))

            end_time[k] = model.array(model.range(0, c), end_time_lambda, 0)

            # Arriving home after max horizon
            home_lateness[k] = model.iif(
                trucks_used[k],
                model.max(0, returning_home_time(k, sequence[c-1],end_time[k][c-1], model, dist_depot,
                                                    nb_breaks, breaks_start_times) - max_horizon),
                0
            )

            # Completing visit after latest end
            late_lambda = model.lambda_function(
                lambda i: model.max(0, end_time[k][i] - latest[sequence[i]]))
            lateness[k] = home_lateness[k] + model.sum(model.range(0, c), late_lambda)

        # Total lateness
        total_lateness = model.sum(lateness)
        #Total number of trucks used
        nb_trucks_used = model.sum(trucks_used)

        # Total distance traveled
        total_distance = model.div(model.round(100 * model.sum(dist_routes)), 100)

        # Objective: minimize the number of trucks used, then minimize the distance traveled
        model.minimize(total_lateness)
        model.minimize(nb_trucks_used)
        model.minimize(total_distance)
        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)
        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - number of trucks used and total distance
        #  - for each truck {trucknumber}: the customers visited [starting and ending service time] | B(starting and ending times)
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("Instance: " + instance_file + "\n")
                f.write("Number of trucks: " + str(nb_trucks_used.value) + " Total distance: " + str(total_distance.value) + " Max horizon: " + str(max_horizon) + "\n")
                f.write("Break frequency: " + str(BREAKFREQUENCY) + " Break duration: " + str(BREAKDURATION) + " Working time: " + str(service_time_data[1]) + "\n")
                f.write("Legend: Client[Start,end] B=Break(Start,end)\n\n")
                for k in range(nb_trucks):
                    if trucks_used[k].value != 1:
                        continue
                    f.write(str(k) + ": ")

                    prev_end_time = 0
                    customer_order = 0
                    customer_end_time = 0
                    customer_start_time = 0
                    for customer in customers_sequences[k].value:
                        customer_end_time = round(end_time[k].value[customer_order])
                        customer_start_time = customer_end_time - service_time_data[customer]

                        # Insert breaks
                        for breakIdx in breaks_start_times[k]:
                            if (breakIdx.value >= prev_end_time and breakIdx.value <= customer_end_time):
                                end_break = breakIdx.value + BREAKDURATION
                                f.write("B(" + str(breakIdx.value) + "," + str(end_break) + ") ")
                        # Values in sequence are in 0...nbCustomers. +1 is to put it back in
                        # 1...nbCustomers+1 as in the data files (0 being the depot)
                        f.write(str(customer + 1) + "[" + str(customer_start_time) + "," + str(customer_end_time) + "] ")

                        prev_end_time = customer_end_time
                        customer_order += 1

                    # Insert break if needed before returning to depot
                    depot_arriving_time = prev_end_time + dist_depot_data[customers_sequences[k].value[customer_order - 1]]
                    for breakIdx in breaks_start_times[k]:
                        if (breakIdx.value >= prev_end_time and breakIdx.value <= depot_arriving_time):
                            end_break = breakIdx.value + BREAKDURATION
                            f.write("B(" + str(breakIdx.value) + "," + str(end_break) + ") ")
                            depot_arriving_time += BREAKDURATION

                    f.write("| ")
                    for breakIdx in breaks_start_times[k]:
                        if (breakIdx.value > depot_arriving_time):
                            f.write("B(" + str(breakIdx.value) + ")")
                    f.write("\n")

# The input files follow the "Solomon" format
def read_input_cvrptwrb(filename):
    file_it = iter(read_elem(filename))

    for i in range(4):
        next(file_it)

    nb_trucks = int(next(file_it))
    truck_capacity = int(next(file_it))

    for i in range(13):
        next(file_it)

    depot_x = int(next(file_it))
    depot_y = int(next(file_it))

    for i in range(2):
        next(file_it)

    max_horizon = int(next(file_it))

    next(file_it)

    customers_x = []
    customers_y = []
    demands = []
    earliest_start = []
    latest_end = []
    service_time = []

    while True:
        val = next(file_it, None)
        if val is None:
            break
        i = int(val) - 1
        customers_x.append(int(next(file_it)))
        customers_y.append(int(next(file_it)))
        demands.append(int(next(file_it)))
        ready = int(next(file_it))
        due = int(next(file_it))
        stime = int(next(file_it))
        earliest_start.append(ready)
        # in input files due date is meant as latest start time
        latest_end.append(due + stime)
        service_time.append(stime)

    nb_customers = i + 1

    # Compute distance matrix
    distance_matrix = compute_distance_matrix(customers_x, customers_y)
    distance_depots = compute_distance_depots(depot_x, depot_y, customers_x, customers_y)

    return nb_customers, nb_trucks, truck_capacity, distance_matrix, distance_depots, \
        demands, service_time, earliest_start, latest_end, max_horizon


# Computes the distance matrix
def compute_distance_matrix(customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_matrix = [[None for i in range(nb_customers)] for j in range(nb_customers)]
    for i in range(nb_customers):
        distance_matrix[i][i] = 0
        for j in range(nb_customers):
            dist = compute_dist(customers_x[i], customers_x[j],
                                customers_y[i], customers_y[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    return distance_matrix


# Computes the distances to depot
def compute_distance_depots(depot_x, depot_y, customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_depots = [None] * nb_customers
    for i in range(nb_customers):
        dist = compute_dist(depot_x, customers_x[i], depot_y, customers_y[i])
        distance_depots[i] = dist
    return distance_depots


def compute_dist(xi, xj, yi, yj):
    return math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2))

    # Sub functions for modelling
def next_available_time(customer, t, model, earliest):
    return model.max(t, earliest[customer])

def needs_break(break_start, start, end, model):
    return model.and_(start <= break_start, end > break_start)

# Next 3 functions compute the different times, taking breaks into account

def travel_end(vehicle, i, time, customers_sequences, model, dist_depot, dist_matrix,
               nb_breaks, breaks_start_times):
    # Compute travel end time
    sequence = customers_sequences[vehicle]
    travel_duration = model.iif(i == 0, dist_depot[sequence[0]], dist_matrix[sequence[i-1]][sequence[i]])
    travel_end = time + travel_duration
    end_with_breaks = travel_end
    for p in range(nb_breaks):
        end_with_breaks = model.iif(needs_break(breaks_start_times[vehicle][p], time, end_with_breaks, model),
                                    end_with_breaks + BREAKDURATION,
                                    end_with_breaks)
    return end_with_breaks

def waiting_and_service_end(vehicle, customer, time, model, earliest, service_time,
                            nb_breaks, breaks_start_times):
    # Compute waiting and service end time
    next_start_without_break = next_available_time(customer, time, model, earliest)
    end_without_break = next_start_without_break + service_time[customer]
    end_with_breaks = end_without_break
    for p in range(nb_breaks):
        end_with_breaks = model.iif(needs_break(breaks_start_times[vehicle][p], time, end_with_breaks, model),
                                    next_available_time(customer, breaks_start_times[vehicle][p] + BREAKDURATION, model, earliest)
                                            + service_time[customer],
                                    end_with_breaks)
    return end_with_breaks

def returning_home_time(vehicle, customer, time, model, dist_depot, nb_breaks, breaks_start_times):
    # Compute returning home time
    end_without_break = time + dist_depot[customer]
    end_with_breaks = end_without_break
    for p in range(nb_breaks):
        end_with_breaks = model.iif(needs_break(breaks_start_times[vehicle][p], time, end_with_breaks, model),
                                    end_with_breaks + BREAKDURATION,
                                    end_with_breaks)
    return end_with_breaks

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python cvrptwrb.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"
    main(instance_file, str_time_limit, output_file)
